# 01 — Financial PhraseBank Exploration

EDA on the Financial PhraseBank dataset (Malo et al., 2014) — financial news sentences annotated by domain experts with negative / neutral / positive labels.

Goals:
- Understand class balance and label distribution
- Sentence length statistics (informs tokenizer `max_length`)
- Token-level patterns by class

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter

from src.data import load_phrasebank, class_distribution

sns.set_theme(style='whitegrid')

In [ ]:
df = load_phrasebank()
print(f"Loaded {len(df):,} sentences")
df.head(5)

## Class distribution

In [ ]:
dist = class_distribution(df)
print(dist.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
dist.plot.bar(ax=ax, color=['#E74C3C', '#95A5A6', '#27AE60'])
ax.set_title('Class distribution (Financial PhraseBank, 75% agreement)')
ax.set_ylabel('Fraction of sentences')
plt.tight_layout()
plt.show()

## Sentence length distribution

In [ ]:
df['n_words'] = df['text'].str.split().str.len()
df['n_chars'] = df['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['n_words'].plot.hist(bins=40, ax=axes[0], color='#3498DB')
axes[0].set_title('Words per sentence')
axes[0].axvline(df['n_words'].quantile(0.95), color='red', ls='--', label='95th pct')
axes[0].legend()

df['n_chars'].plot.hist(bins=40, ax=axes[1], color='#9B59B6')
axes[1].set_title('Characters per sentence')
plt.tight_layout()
plt.show()

print(df[['n_words', 'n_chars']].describe().round(1).to_string())

## Most common words per class (after stopword filtering)

In [ ]:
import re

STOPWORDS = set("""the a an and or but if to of in on for with by at from as is was were be been being have has had do does did this that these those it its his her their our your""".split())

def top_words(texts, n=20):
    counter = Counter()
    for t in texts:
        for w in re.findall(r"[a-zA-Z]+", t.lower()):
            if w not in STOPWORDS and len(w) > 2:
                counter[w] += 1
    return counter.most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (label, color) in zip(axes, [('negative', '#E74C3C'), ('neutral', '#95A5A6'), ('positive', '#27AE60')]):
    sub = df[df['label_name'] == label]
    words = top_words(sub['text'], n=15)
    pd.Series(dict(words)).sort_values().plot.barh(ax=ax, color=color)
    ax.set_title(f'Top words — {label} ({len(sub):,} sentences)')
plt.tight_layout()
plt.show()

## Sample sentences per class

In [ ]:
for label in ['negative', 'neutral', 'positive']:
    print(f"\n=== {label.upper()} ===")
    for t in df[df['label_name'] == label].sample(3, random_state=42)['text']:
        print(f"  - {t}")

## Takeaways

- **Class imbalance**: neutral class dominates (~60%), negative is the smallest (~13%). Use **macro F1** as the headline metric so the rare classes count equally.
- **Sentence length**: 95% of sentences fit in ~30 words, so `max_length=128` for the tokenizer is comfortably oversized but cheap.
- **Vocabulary signal**: positive sentences feature growth/profit/increase verbs; negatives feature loss/decrease/decline. This is a textbook fine-tuning target for a transformer.